1. Librerias

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

2. Carga de datos

In [2]:
df = pd.read_csv('../data/clientes.csv')
df.shape

(7043, 21)

3. Limpieza de Total Charge
Cambiar totalcharge a valor numerico

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['TotalCharges'].dtype

dtype('float64')

4. Elimiar 'Customer ID'
No aporta informacion predictiva y puede causar overfitting

In [4]:
df = df.drop(columns=['customerID'])
df.shape

(7043, 20)

5. Churn
Convertir Yes/No a 1/0 para que el modelo lo pueda leer

In [5]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

6. Simplificar categorias redundantes
'Online Security' tiene infromacion como Yes/No/No Internet Service. 

In [6]:
cols_simplificar = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in cols_simplificar:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

df['OnlineSecurity'].unique()

array(['No', 'Yes'], dtype=object)

7. Codificar categorias binarias
Mismo paso que con Churn: cambiar todo a 1/0
Yes/Male = 1
No/Female = 0

In [7]:
cols_binarias = ['gender', 'Partner', 'Dependents', 'PhoneService',
                  'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                  'DeviceProtection', 'TechSupport', 'StreamingTV',
                  'StreamingMovies', 'PaperlessBilling']

for col in cols_binarias:
    df[col] = df[col].map(lambda x: 1 if x in ['Yes', 'Male'] else 0)

df[cols_binarias].head()

,gender,Partner,Dependents,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling
0,0,1,0,0,0,0,1,0,0,0,0,1
1,1,0,0,1,0,1,0,1,0,0,0,0
2,1,0,0,1,0,1,1,0,0,0,0,1
3,1,0,0,0,0,1,0,1,1,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,1


8. Codificar variables categoricas con mas de 2 valores usando label encoding
Para la fase de modelado usaremos modelos basados en arboles (RF, XGB)

In [8]:
from sklearn.preprocessing import LabelEncoder

cols_multicategoria = ['Contract', 'PaymentMethod', 'InternetService']
encoders = {}

for col in cols_multicategoria:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

Contract: {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
PaymentMethod: {'Bank transfer (automatic)': 0, 'Credit card (automatic)': 1, 'Electronic check': 2, 'Mailed check': 3}
InternetService: {'DSL': 0, 'Fiber optic': 1, 'No': 2}


9. Separar train/test y estandarizar variables numéricas

Separamos primero en train/test, y ajustamos el escalador (`fit`) solo con los datos de entrenamiento. Si ajustáramos el escalador con todo el dataset antes de separar, la media y desviación estándar usadas "verían" datos de test, filtrando información hacia el conjunto que se supone que el modelo no conoce (data leakage).

In [9]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.shape, X_test.shape

((5634, 19), (1409, 19))

10. guardar los datos procesados

In [10]:
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("Guardado correctamente")

Guardado correctamente
